In [1]:
import cv2
from cv2 import aruco
import numpy as np
import msgpack as mp
import msgpack_numpy as mpn
import os
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from joblib import Parallel, delayed

### Defining board dimentions

In [2]:
patternSize = (8, 12)
squareSize = 30

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

def construct3DPoints(patternSize, squareSize):
    X = np.zeros((patternSize[0] * patternSize[1], 3), np.float32)
    X[:, :2] = np.mgrid[0 : patternSize[0], 0 : patternSize[1]].T.reshape(-1, 2)
    X = X * squareSize
    return X

boardPoints = construct3DPoints(patternSize, squareSize)

### Path defs

In [5]:
from pathlib import Path

# Path defs
project_root = Path.cwd().parents[2]
data_root = "data"
recording_type = 'calibration'
camera_type = 'dual_160'
calib_folder_name = "dual_cam_calibration_checker_sz_30mm"

calib_data_folder = os.path.join(
    project_root, data_root, recording_type, camera_type, calib_folder_name
)
cam_ov9281_data = os.path.join(calib_data_folder, "cam1_ov9281.msgpack")
cam_ov9281_meta = os.path.join(calib_data_folder, "cam1_timestamp.msgpack")

cam_imx219_data = os.path.join(calib_data_folder, "cam0_imx219.msgpack")
cam_imx219_meta = os.path.join(calib_data_folder, "cam0_timestamp.msgpack")
os.path.exists(cam_imx219_meta)

True

### Parse metadata

In [6]:
def get_metadata(metaf):
    f = open(metaf, 'rb')
    _ = np.array(list(mp.Unpacker(f, object_hook=mpn.decode)))
    sync, timestamps = _[:,0], _[:,1]
    sync = sync.astype(int).astype(bool)
    timestamp_dt = timestamps.astype('datetime64[us]')
    return sync, timestamp_dt

sync_cam0, timestamp_cam0 = get_metadata(cam_imx219_meta)
sync_cam1, timestamp_cam1 = get_metadata(cam_ov9281_meta)

In [12]:
duration = timestamp_cam0[-1] - timestamp_cam0[0]
# duration in seconds
duration_s = duration.astype('timedelta64[s]').astype(float)
print(f"Duration of recording: {duration_s:.2f} seconds")

Duration of recording: 142.00 seconds


### Get video unpacker file

In [56]:
def get_video_unpacker(vidf):
    _video_file = open(vidf, "rb")
    _video_data = mp.Unpacker(_video_file, object_hook=mpn.decode)
    return _video_data

cam0_upak = get_video_unpacker(cam_imx219_data)
cam1_upak = get_video_unpacker(cam_ov9281_data)

In [ ]:
# next(cam0_upak).shape

(1232, 1640, 3)

### Corner detection for both cameras

In [31]:
# calibration
def detectCorners(data):
    # _frame = cv2.rotate(_frame.copy(), cv2.ROTATE_180)
    frame_id, _frame = data
    if len(_frame.shape) == 3:
        _frame = cv2.cvtColor(_frame, cv2.COLOR_RGB2GRAY)
    ret, corners = cv2.findChessboardCorners(_frame, patternSize)
    if ret:
        corners = cv2.cornerSubPix(
            _frame,
            corners,
            (5, 5),
            (-1, -1),
            criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001),
        )
    else:
        return None, frame_id
    return corners, frame_id

cam0_results = Parallel(n_jobs=20, verbose=0)(
    delayed(detectCorners)(frame) for frame in tqdm(enumerate(cam0_upak))
)

cam1_results = Parallel(n_jobs=20, verbose=0)(
    delayed(detectCorners)(frame) for frame in tqdm(enumerate(cam1_upak))
)

0it [00:00, ?it/s]

0it [00:00, ?it/s]

### Remove in-homogenious parts

In [42]:
cam0_cb_corners = {
    'corners':[],
    'frame_idx':[],
    'timestamp':[],
    'sync':[]
}

cam1_cb_corners = {
    'corners':[],
    'frame_idx':[],
    'timestamp':[],
    'sync':[]
}

for corners, frame_idx in cam0_results:
    if corners is not None and len(corners) == 96:
        cam0_cb_corners['corners'].append(corners)
        cam0_cb_corners['frame_idx'].append(frame_idx)
        cam0_cb_corners['timestamp'].append(timestamp_cam0[frame_idx])
        cam0_cb_corners['sync'].append(timestamp_cam0[frame_idx])

for corners, frame_idx in cam1_results:
    if corners is not None and len(corners) == 96:
        cam1_cb_corners['corners'].append(corners)
        cam1_cb_corners['frame_idx'].append(frame_idx)
        cam1_cb_corners['timestamp'].append(timestamp_cam1[frame_idx])
        cam1_cb_corners['sync'].append(timestamp_cam1[frame_idx])

### Writing to files

In [51]:
# Get the directory of the video file
def write_to_file(corners_dict, camera):
    import pickle
    corners_file = os.path.join(calib_data_folder, f"chessb_corners_{camera}.pkl")
    with open(corners_file, "wb") as f:
        pickle.dump(corners_dict, f)

write_to_file(cam0_cb_corners, 'cam0_imx219')
write_to_file(cam1_cb_corners, 'cam1_ov9281')